In [1]:
# Importing necessary libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import math
from scipy.spatial.transform import Rotation as R
from scipy.linalg import block_diag
import timeit as timee
import import_ipynb
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection
import matplotlib.collections
#import effort_solver_nn_3_triangles_3D_stiffness_information_Classical_FEM_changed as solv
import Functions_for_material_law as ML
from sklearn.preprocessing  import MinMaxScaler
from datetime import datetime
start_time = datetime.now()
from mpl_toolkits.mplot3d import Axes3D
import sympy as sym
from matplotlib.patches import Rectangle
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Flatten, Dense, Conv2D, MaxPool2D, Dropout,LSTM,Embedding,Layer, LSTM, Conv1D, MaxPool1D, TimeDistributed, Input, GRU
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras import metrics
from tensorflow.keras.callbacks import Callback
#import plotly.graph_objects as go
import pickle
from tensorflow.keras.models import Model, load_model
%matplotlib qt

importing Jupyter notebook from Functions_for_material_law.ipynb


In [2]:
def props_update(mat_params, i,l,j,k, pois):
    
    return (mat_params['E'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]*pois)/((1 + pois)*(1 - 2*pois)), 0.5*mat_params['E'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]/(1 + pois)


def tens_to_voigt(tens):
    
    stress    = np.zeros(shape=(6,))
    
    stress[0] = tens[0][0]
    stress[1] = tens[1][1]
    stress[2] = tens[2][2]
    stress[3] = tens[1][2]
    stress[4] = tens[0][2]
    stress[5] = tens[0][1]
    
    return stress


def voigt_to_tens(stress):
    
    tens = np.zeros(shape=(3,3))
    
    
    tens[0][0] = stress[0]
    tens[1][1] = stress[1]
    tens[2][2] = stress[2]
    tens[0][1] = stress[5]
    tens[0][2] = stress[4]
    tens[1][0] = stress[5]
    tens[1][2] = stress[3]
    tens[2][0] = stress[4]
    tens[2][1] = stress[3]
    
    
    return tens


def split(tens):
   
    I      = np.eye(3,3)
    
    aa     = voigt_to_tens(tens)
    
    shydro = (1/3*(aa.trace()))
    
    return aa - shydro*I, shydro*I


In [3]:
# Initializing all the material properties

mat_params    = {}
I             = np.eye(3,3)
pois          = 0.32
timeincrement = 1e-5
N             = 4.22

for i in range(1):
    
    for l in range(1):
    
        for j in range(1):

            for k in range(1):


                mat_params['stress_eqv_trial'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  = 0 
                mat_params['stress_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = np.zeros(shape=(3,3))
                mat_params['flow'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                              = np.zeros(shape=(3,3))
                mat_params['eps_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                          = np.zeros(shape=(3,3))
                mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = np.zeros(shape=(3,3))
                mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = np.zeros(shape=(3,3))
                mat_params['eps'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                               = np.zeros(shape=(3,3))
                mat_params['stress_n'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                          = np.zeros(shape=(6,))
                mat_params['stress_yield'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                      = 167.88
                mat_params['E'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                 = 198600
                mat_params['delta_eps'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                         = np.zeros(shape=(6,))
                mat_params['delta_eps_cor'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                     = np.zeros(shape=(6,))
                #mat_params['strain_n_1']                                                   = np.zeros(shape=(6,))
                mat_params['lam'+str(i+1)+str(l+1)+str(j+1)+str(k+1)], mat_params['mu'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]   = props_update(mat_params, i,l,j,k, pois) 
                mat_params['pois'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                              = pois
                mat_params['stress_n-1'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = np.zeros(shape=(6,))
                mat_params['eps_vec'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                           = np.zeros(shape=(6,))
                aaa, bbb = mat_params['lam'+str(i+1)+str(l+1)+str(j+1)+str(k+1)], mat_params['mu'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]
                aa = 2*bbb + aaa
                mat_params['stress_K'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                          = 63.12
                mat_params['abs_stress'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = []
                mat_params['stress_back'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = np.zeros(shape=(3,3))
                mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                   = np.zeros(shape=(3,3))
                mat_params['abs_strain'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = []
                mat_params['C'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                 = np.array([[aa, aaa, aaa, 0, 0, 0], 
                           [aaa, aa, aaa, 0, 0, 0],
                           [aaa, aaa, aa, 0, 0, 0],
                          [0, 0, 0, bbb,  0,  0],
                          [0, 0, 0, 0,   bbb, 0, ],
                          [0, 0, 0, 0, 0, bbb]])
                mat_params['C_evp_fourth'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                               = np.zeros(shape=(3,3,3,3))
                mat_params['C_evp'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                      = mat_params['C'+str(i+1)+str(l+1)+str(j+1)+str(k+1)].copy()
                mat_params['abs_eqpl'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                   = []                                              
                mat_params['C_ep'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                                    = np.array([[aa, aaa, aaa, 0, 0, 0], 
                           [aaa, aa, aaa, 0, 0, 0],
                           [aaa, aaa, aa, 0, 0, 0],
                          [0, 0, 0, bbb,  0,  0],
                          [0, 0, 0, 0,   bbb, 0, ],
                          [0, 0, 0, 0, 0, bbb]])

                mat_params['EQPLAS'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                   = np.sqrt((np.tensordot(mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)],mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]))*(2/3))
                K                                                                          = mat_params['E'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]/(1 - 2*pois)
                mat_params['Delta_d'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  = 0.0
                mat_params['R'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = 0.0
                mat_params['b1'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = 8.
                mat_params['b2'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = 60.
                mat_params['D'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = 0.0
                mat_params['D_cr'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                     = 0.45
                mat_params['e_d'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                      = 0.0
                mat_params['e_r'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                      = 0.155
            #mat_params['stiff_point'+str(i+1)+str(l+1)+str(j+1)]                           = np.zeros(shape=(DOF_per_node*nodes_per_element, DOF_per_node*nodes_per_element))
    
    
    #ax['element'+str(i+1)]                                                    = np.zeros(shape=(DOF_per_node*nodes_per_element ,1))
    
#     nn['element_force'+str(i+1)]                                              = []
#     nn['element_disp'+str(i+1)]                                               = []
#     nn['element_stiff'+str(i+1)]                                              = []

In [4]:
delta_st = []
for uu in range(12000):
    
    if uu<2000:
        
        delta_st.append(np.array([-1.13662218e-04,  1.37489468e-07,  5.34234067e-05,  1.94601544e-06,
        2.55335799e-06,  6.21437461e-07]))
        
    elif uu>=2000 and uu<4000:
        
        delta_st.append(1*np.array([ 1.80716181e-05,  3.78134516e-05, -2.62988560e-05, -1.56952564e-04,-3.58919652e-05, -1.02044028e-05])*0.01)
    
    elif uu>=4000 and uu<6000:
        
        delta_st.append(1*np.array([ 1.80716181e-05,  3.78134516e-05, -2.62988560e-05, -1.56952564e-04,-3.58919652e-05, -1.02044028e-05])*0.01)     
        
        
    elif uu>6000 and uu<=8000:
        
        delta_st.append(1*np.array([ 1.80716181e-05,  3.78134516e-05, -2.62988560e-05, -1.56952564e-04,-3.58919652e-05, -1.02044028e-05])*0.01)        
    elif uu>8000 and uu<=10000:
        
        delta_st.append(1*np.array([ 1.80716181e-05,  3.78134516e-05, -2.62988560e-05, -1.56952564e-04,-3.58919652e-05, -1.02044028e-05])*0.01)     
        
        
    elif uu>10000 and uu<=12000:
        
        delta_st.append(1*np.array([ 1.80716181e-05,  3.78134516e-05, -2.62988560e-05, -1.56952564e-04,-3.58919652e-05, -1.02044028e-05])*0.01)   
        

    else:
        
        delta_st.append(1*np.array([ 1.80716181e-05,  3.78134516e-05, -2.62988560e-05, -1.56952564e-04,-3.58919652e-05, -1.02044028e-05])*0.01)   
        
        

In [5]:
@tf.function
def train_LSTM_static(x_batch_train, stress_dev_trial, stress_back_trial, N_dirr, timeincrement, mat_params, N, state_inputs, model):

    with tf.GradientTape(persistent=True) as tt:
        F_n, h, c    = model([x_batch_train]+[state_inputs])
        tt.watch(model.trainable_weights)
        sigma_star_N = stress_dev_trial - 3*(mat_params['mu'+str(i+1)+str(m+1)+str(n+1)]*F_n*(N_dirr)*(timeincrement))
        X_star_N     = (stress_back_trial+ ((3100)*F_n*timeincrement*(N_dirr)))/(1 + 110*F_n*timeincrement) 
        J2_N         = tf.math.sqrt(1.5*tf.tensordot(sigma_star_N[0] - X_star_N[0], sigma_star_N[0]-X_star_N[0], axes=2))
        RHS_N        = J2_N - tf.math.pow(F_n,(1/N))*(mat_params['stress_K'+str(i+1)+str(m+1)+str(n+1)]) - mat_params['stress_yield'+str(i+1)+str(m+1)+str(n+1)]
        loss_value   = tf.reduce_mean(tf.math.pow(RHS_N*tf.ones(shape=(1,1)),2))
        #loss_value   = tf.experimental.numpy.log10(tf.math.abs(loss_value1) + tf.ones(shape=(1,1)))

    grads_batch      = tt.gradient(loss_value, model.trainable_weights)
    opt.apply_gradients(zip(grads_batch,model.trainable_weights))

    
    return loss_value, F_n, h, c

In [6]:
def implicit_integration_vp_1_exp(mat_params, i, N, timeincrement, number_of_iteration, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n, N_dirr):
    
    Delta_pl_L    =   np.zeros(shape=(2,1,1))
    
    Delta_pl_U1   =   (((stress_eqv_trial - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])**(N)).numpy()

    Delta_pl_U    =  Delta_pl_U1
    
    return Delta_pl_U*timeincrement, mat_params

In [7]:
def implicit_integration_vp(mat_params, i, N, timeincrement, number_of_iteration, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n, N_dirr):
    

    Delta_pl_L  = 0
    Delta_pl_U1 = (((stress_eqv_trial - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])**(N))*timeincrement
    #Delta_pl_U1 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial-mat_params['stress_yield'+str(i+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(m+1)+str(n+1)]))
    mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
    stress_dev_trial_n  =   mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-I*(1/3*mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].trace())
    stress_eqv_trial_n  = np.sqrt(1.5*(np.tensordot(stress_dev_trial_n - stress_back_trial, stress_dev_trial-stress_back_trial)))
    Delta_pl_U2 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial_n-mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))



    if wahl==0:

        Delta_pl_U = Delta_pl_U2
        #print('Delta_pl_U1 is chosen')

    else:
        Delta_pl_U = Delta_pl_U2 
        #print('Delta_pl_U2 is chosen')

    #print(Delta_pl_L, Delta_pl_U1, Delta_pl_U2, Delta_pl_U)

    for q in range(5000):

        h_L                      =         mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(Delta_pl_L/timeincrement)**(1/N) 
        Z_L                      =         stress_dev_trial - ((mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(1 + 110*Delta_pl_L))
        alpha_L                  =         1 + (3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (3100/(1 + 110*Delta_pl_L)))*(Delta_pl_L/h_L)
        J_L                      =         np.sqrt(1.5*np.tensordot(Z_L, Z_L))
        RHS_L                    =         (1/alpha_L)*(J_L) - h_L

        h_U                      =         mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(Delta_pl_U/timeincrement)**(1/N) 
        Z_U                      =         stress_dev_trial - ((mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(1 + 110*Delta_pl_U))
        alpha_U                  =         1 + (3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (3100/(1 + 110*Delta_pl_U)))*(Delta_pl_U/h_U)
        J_U                      =         np.sqrt(1.5*np.tensordot(Z_U, Z_U))
        RHS_U                    =         (1/alpha_U)*(J_U) - h_U

        Delta_pl_N               =         Delta_pl_U - (RHS_U*(Delta_pl_L-Delta_pl_U))/(RHS_L - RHS_U)
        #print(Delta_pl_N)

        h_N                      =         mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(Delta_pl_N/timeincrement)**(1/N) 
        Z_N                      =         stress_dev_trial - ((mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(1 + 110*Delta_pl_N))
        alpha_N                  =         1 + (3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (3100/(1 + 110*Delta_pl_N)))*(Delta_pl_N/h_N)
        J_N                      =         np.sqrt(1.5*np.tensordot(Z_N, Z_N))
        RHS_N                    =         (1/alpha_N)*(J_N) - h_N


        #print(RHS_N)
        if abs(RHS_N)<1e-10:

            Delta_pl = Delta_pl_N
            #print(Delta_pl*timeincrement)
            #print('Your increment number '+str(k+1)+' has converged with '+str(q+1)+' iterations')

            break

        elif RHS_N*RHS_L<0:

            Delta_pl_L = Delta_pl_L
            Delta_pl_U = Delta_pl_N

        elif RHS_N*RHS_U<0:
            Delta_pl_L = Delta_pl_N
            Delta_pl_U = Delta_pl_U
                
                
    return Delta_pl_N, alpha_N, h_N, mat_params, q+1
    


In [8]:
def implicit_integration_vp_1(mat_params, i, N, timeincrement, number_of_iteration, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n, N_dirr):
    

    Delta_pl_L  = 0
    Delta_pl_U1 = (((stress_eqv_trial - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])**(N))
    #Delta_pl_U1 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial-mat_params['stress_yield'+str(i+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(m+1)+str(n+1)]))
    mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
    stress_dev_trial_n  =   mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-I*(1/3*mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].trace())
    stress_eqv_trial_n  = np.sqrt(1.5*(np.tensordot(stress_dev_trial_n - stress_back_trial, stress_dev_trial-stress_back_trial)))
    Delta_pl_U2 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial_n-mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
    Delta_pl_U2 = Delta_pl_U2/timeincrement


    if Delta_pl_U1<Delta_pl_U2:

        Delta_pl_U = Delta_pl_U1
        #print('Delta_pl_U1 is chosen')

    else:
        Delta_pl_U = Delta_pl_U2
        #print('Delta_pl_U2 is chosen')

    #print(Delta_pl_L, Delta_pl_U1, Delta_pl_U2, Delta_pl_U)

    for q in range(5000):


        sigma_star_L = stress_dev_trial - 3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_L*(N_dirr)*(timeincrement))
        X_star_L     = (stress_back_trial+ ((3100)*Delta_pl_L*timeincrement*(N_dirr)))/(1 + 110*Delta_pl_L*timeincrement) 
        J2_L         = np.sqrt(1.5*np.tensordot(sigma_star_L - X_star_L, sigma_star_L-X_star_L))
        RHS_L        = J2_L - (Delta_pl_L)**((1/N))*(mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]
        sigma_star_U = stress_dev_trial - 3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_U*(N_dirr)*(timeincrement))
        X_star_U     = (stress_back_trial+ ((3100)*Delta_pl_U*timeincrement*(N_dirr)))/(1 + 110*Delta_pl_U*timeincrement) 
        J2_U         = np.sqrt(1.5*np.tensordot(sigma_star_U - X_star_U, sigma_star_U-X_star_U))
        RHS_U        = J2_U - (Delta_pl_U)**((1/N))*(mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]
        Delta_pl_N   = Delta_pl_U - (RHS_U*(Delta_pl_L-Delta_pl_U))/(RHS_L - RHS_U)
        sigma_star_N = stress_dev_trial - 3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_N*(N_dirr)*(timeincrement))
        X_star_N     = (stress_back_trial+ ((3100)*Delta_pl_N*timeincrement*(N_dirr)))/(1 + 110*Delta_pl_N*timeincrement) 
        J2_N         = np.sqrt(1.5*np.tensordot(sigma_star_N - X_star_N, sigma_star_N-X_star_N))
        RHS_N        = J2_N - (Delta_pl_N)**((1/N))*(mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]
        #print(RHS_N)
        if abs(RHS_N)<1e-6:

            Delta_pl = Delta_pl_N
            #if sequence_tracker==0:
            #    deqpl1.append(Delta_pl)
            #elif sequence_tracker==1:
            #    deqpl2.append(Delta_pl)
            #print('In the converged zone')
            #print('Your increment number '+str(t+1)+' has converged with '+str(q+1)+' iterations')

            break

        elif RHS_N*RHS_L<0:

            Delta_pl_L = Delta_pl_L
            Delta_pl_U = Delta_pl_N

        elif RHS_N*RHS_U<0:
            Delta_pl_L = Delta_pl_N
            Delta_pl_U = Delta_pl_U
                
                
    return Delta_pl*timeincrement, mat_params, q+1, sigma_star_N

In [9]:
def implicit_integration_vp_2(mat_params, i, N, timeincrement, number_of_iteration, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n, N_dirr):
    

    Delta_pl_L  = 0
    Delta_pl_U1 = (((stress_eqv_trial - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])**(N))
    #Delta_pl_U1 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial-mat_params['stress_yield'+str(i+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(m+1)+str(n+1)]))
    mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
    stress_dev_trial_n  =   mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-I*(1/3*mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].trace())
    stress_eqv_trial_n  = np.sqrt(1.5*(np.tensordot(stress_dev_trial_n - stress_back_trial, stress_dev_trial-stress_back_trial)))
    Delta_pl_U2 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial_n-mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
    Delta_pl_U2 = Delta_pl_U2/timeincrement


    if Delta_pl_U1<Delta_pl_U2:

        Delta_pl_U = Delta_pl_U1
        #print('Delta_pl_U1 is chosen')

    else:
        Delta_pl_U = Delta_pl_U2
        #print('Delta_pl_U2 is chosen')

    #print(Delta_pl_L, Delta_pl_U1, Delta_pl_U2, Delta_pl_U)

    for q in range(5000):


        sigma_star_L = stress_dev_trial - 3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_L*(N_dirr)*(timeincrement))
        X_star_L     = (stress_back_trial+ ((3100)*Delta_pl_L*timeincrement*(N_dirr)))/(1 + 110*Delta_pl_L*timeincrement) 
        J2_L         = np.sqrt(1.5*np.tensordot(sigma_star_L - X_star_L, sigma_star_L-X_star_L))
        R_L          = mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*Delta_pl_L*timeincrement
        RHS_L        = J2_L - (Delta_pl_L)**((1/N))*(mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]
        sigma_star_U = stress_dev_trial - 3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_U*(N_dirr)*(timeincrement))
        X_star_U     = (stress_back_trial+ ((3100)*Delta_pl_U*timeincrement*(N_dirr)))/(1 + 110*Delta_pl_U*timeincrement) 
        J2_U         = np.sqrt(1.5*np.tensordot(sigma_star_U - X_star_U, sigma_star_U-X_star_U))
        R_U          = mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*Delta_pl_U*timeincrement
        RHS_U        = J2_U - (Delta_pl_U)**((1/N))*(mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]
        Delta_pl_N   = Delta_pl_U - (RHS_U*(Delta_pl_L-Delta_pl_U))/(RHS_L - RHS_U)
        sigma_star_N = stress_dev_trial - 3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_N*(N_dirr)*(timeincrement))
        X_star_N     = (stress_back_trial+ ((3100)*Delta_pl_N*timeincrement*(N_dirr)))/(1 + 110*Delta_pl_N*timeincrement) 
        J2_N         = np.sqrt(1.5*np.tensordot(sigma_star_N - X_star_N, sigma_star_N-X_star_N))
        R_N          = mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*Delta_pl_N*timeincrement
        RHS_N        = J2_N - (Delta_pl_N)**((1/N))*(mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]
        #print(RHS_N)
        if abs(RHS_N)<1e-6:

            Delta_pl = Delta_pl_N
            #if sequence_tracker==0:
            #    deqpl1.append(Delta_pl)
            #elif sequence_tracker==1:
            #    deqpl2.append(Delta_pl)
            #print('In the converged zone')
            #print('Your increment number '+str(t+1)+' has converged with '+str(q+1)+' iterations')

            break

        elif RHS_N*RHS_L<0:

            Delta_pl_L = Delta_pl_L
            Delta_pl_U = Delta_pl_N

        elif RHS_N*RHS_U<0:
            Delta_pl_L = Delta_pl_N
            Delta_pl_U = Delta_pl_U
                
                
    return Delta_pl*timeincrement, mat_params, q+1, sigma_star_N

In [10]:
def implicit_integration_vp_2_dam(mat_params, i, N, timeincrement, number_of_iteration, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,stress_sph,l,m,n, N_dirr):
    

    Delta_pl_L  = 0
    Delta_pl_U1 = ((((stress_eqv_trial/(1 -  mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])**(N))
    #Delta_pl_U1 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial-mat_params['stress_yield'+str(i+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(m+1)+str(n+1)]))
    mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
    stress_dev_trial_n  =   mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-I*(1/3*mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].trace())
    stress_eqv_trial_n  = (np.sqrt(1.5*(np.tensordot(stress_dev_trial_n - stress_back_trial, stress_dev_trial-stress_back_trial))))/(1 - mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
    Delta_pl_U2 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial_n-mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
    Delta_pl_U2 = Delta_pl_U2/timeincrement
    sph_        = (1/3)*(stress_sph).trace()
    

    if Delta_pl_U1<Delta_pl_U2:

        Delta_pl_U = Delta_pl_U1
        #print('Delta_pl_U1 is chosen')

    else:
        Delta_pl_U = Delta_pl_U2
        #print('Delta_pl_U2 is chosen')

    #print(Delta_pl_L, Delta_pl_U1, Delta_pl_U2, Delta_pl_U)

    for q in range(5000):


        sigma_star_L = stress_dev_trial - 3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_L*(N_dirr)*(timeincrement))
        X_star_L     = (stress_back_trial+ ((2500)*Delta_pl_L*timeincrement*(N_dirr)))/(1 + 20.30*Delta_pl_L*timeincrement) 
        J2_L         = np.sqrt(1.5*np.tensordot(sigma_star_L - X_star_L, sigma_star_L-X_star_L))
        R_L          = mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*Delta_pl_L*timeincrement
        D_L          = (1) - (mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (1.35)*(Delta_pl_L*timeincrement)*((2/3)*(1+mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) + 
                        3*((sph_/J2_L)**(2))*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))
        RHS_L        = J2_L/(D_L) - (Delta_pl_L)**((1/N))*(mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-R_L
        sigma_star_U = stress_dev_trial - 3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_U*(N_dirr)*(timeincrement))
        X_star_U     = (stress_back_trial+ ((2500)*Delta_pl_U*timeincrement*(N_dirr)))/(1 + 20.30*Delta_pl_U*timeincrement) 
        J2_U         = np.sqrt(1.5*np.tensordot(sigma_star_U - X_star_U, sigma_star_U-X_star_U))
        R_U          = mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*Delta_pl_U*timeincrement
        D_U          = (1) - (mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (1.35)*(Delta_pl_U*timeincrement)*((2/3)*(1+mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) + 
                                     3*((sph_/J2_U)**(2))*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))
        RHS_U        = J2_U/(D_U) - (Delta_pl_U)**((1/N))*(mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - R_U
        Delta_pl_N   = Delta_pl_U - (RHS_U*(Delta_pl_L-Delta_pl_U))/(RHS_L - RHS_U)
        sigma_star_N = stress_dev_trial - 3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_N*(N_dirr)*(timeincrement))
        X_star_N     = (stress_back_trial+ ((2500)*Delta_pl_N*timeincrement*(N_dirr)))/(1 + 20.30*Delta_pl_N*timeincrement) 
        J2_N         = np.sqrt(1.5*np.tensordot(sigma_star_N - X_star_N, sigma_star_N-X_star_N))
        R_N          = mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*Delta_pl_N*timeincrement
        D_N          = (1) - (mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (1.35)*(Delta_pl_N*timeincrement)*((2/3)*(1+mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) + 
                                     3*((sph_/J2_N)**(2))*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))
        RHS_N        = J2_N/(D_N) - (Delta_pl_N)**((1/N))*(mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - R_N
        #print(RHS_N)
        if abs(RHS_N)<1e-6:

            Delta_pl = Delta_pl_N
            #if sequence_tracker==0:
            #    deqpl1.append(Delta_pl)
            #elif sequence_tracker==1:
            #    deqpl2.append(Delta_pl)
            #print('In the converged zone')
            #print('Your increment number '+str(t+1)+' has converged with '+str(q+1)+' iterations')

            break

        elif RHS_N*RHS_L<0:

            Delta_pl_L = Delta_pl_L
            Delta_pl_U = Delta_pl_N

        elif RHS_N*RHS_U<0:
            Delta_pl_L = Delta_pl_N
            Delta_pl_U = Delta_pl_U
                
                
    return Delta_pl*timeincrement, mat_params, q+1, sigma_star_N, J2_N

In [11]:
def implicit_integration_vp_3_dam(mat_params, i, N, timeincrement, number_of_iteration, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,stress_sph,l,m,n, N_dirr):
    

    Delta_pl_L  = 0
    Delta_pl_U1 = ((((stress_eqv_trial/(1)) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])**(N))
    #Delta_pl_U1 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial-mat_params['stress_yield'+str(i+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(m+1)+str(n+1)]))
    mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
    stress_dev_trial_n  =   mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-I*(1/3*mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].trace())
    stress_eqv_trial_n  = (np.sqrt(1.5*(np.tensordot(stress_dev_trial_n - stress_back_trial, stress_dev_trial-stress_back_trial))))/(1)
    Delta_pl_U2 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial_n-mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
    Delta_pl_U2 = Delta_pl_U2/timeincrement
    sph_        = (1/3)*(stress_sph).trace()
    

    if Delta_pl_U1<Delta_pl_U2:

        Delta_pl_U = Delta_pl_U1
        #print('Delta_pl_U1 is chosen')

    else:
        Delta_pl_U = Delta_pl_U2
        #print('Delta_pl_U2 is chosen')

    #print(Delta_pl_L, Delta_pl_U1, Delta_pl_U2, Delta_pl_U)

    for q in range(5000):


        sigma_star_L = stress_dev_trial - 3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_L*(N_dirr)*(timeincrement))
        X_star_L     = (stress_back_trial+ ((2500)*Delta_pl_L*timeincrement*(N_dirr)))/(1 + 20.30*Delta_pl_L*timeincrement) 
        J2_L         = np.sqrt(1.5*np.tensordot(sigma_star_L - X_star_L, sigma_star_L-X_star_L))
        R_L          = mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*Delta_pl_L*timeincrement
        #D_L          = (1) - (mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (1.35)*(Delta_pl_L*timeincrement)*((2/3)*(1+mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) + 
        #                3*((sph_/J2_L)**(2))*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))
        RHS_L        = J2_L/(1) - (Delta_pl_L)**((1/N))*(mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-R_L
        sigma_star_U = stress_dev_trial - 3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_U*(N_dirr)*(timeincrement))
        X_star_U     = (stress_back_trial+ ((2500)*Delta_pl_U*timeincrement*(N_dirr)))/(1 + 20.30*Delta_pl_U*timeincrement) 
        J2_U         = np.sqrt(1.5*np.tensordot(sigma_star_U - X_star_U, sigma_star_U-X_star_U))
        R_U          = mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*Delta_pl_U*timeincrement
        #D_U          = (1) - (mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (1.35)*(Delta_pl_U*timeincrement)*((2/3)*(1+mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) + 
        #                             3*((sph_/J2_U)**(2))*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))
        RHS_U        = J2_U/(1) - (Delta_pl_U)**((1/N))*(mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - R_U
        Delta_pl_N   = Delta_pl_U - (RHS_U*(Delta_pl_L-Delta_pl_U))/(RHS_L - RHS_U)
        sigma_star_N = stress_dev_trial - 3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_N*(N_dirr)*(timeincrement))
        X_star_N     = (stress_back_trial+ ((2500)*Delta_pl_N*timeincrement*(N_dirr)))/(1 + 20.30*Delta_pl_N*timeincrement) 
        J2_N         = np.sqrt(1.5*np.tensordot(sigma_star_N - X_star_N, sigma_star_N-X_star_N))
        R_N          = mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*Delta_pl_N*timeincrement
        #D_N          = (1) - (mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (1.35)*(Delta_pl_N*timeincrement)*((2/3)*(1+mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) + 
        #                             3*((sph_/J2_N)**(2))*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))
        RHS_N        = J2_N/(1) - (Delta_pl_N)**((1/N))*(mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - R_N
        #print(RHS_N)
        if abs(RHS_N)<1e-6:

            Delta_pl = Delta_pl_N
            #if sequence_tracker==0:
            #    deqpl1.append(Delta_pl)
            #elif sequence_tracker==1:
            #    deqpl2.append(Delta_pl)
            #print('In the converged zone')
            #print('Your increment number '+str(t+1)+' has converged with '+str(q+1)+' iterations')

            break

        elif RHS_N*RHS_L<0:

            Delta_pl_L = Delta_pl_L
            Delta_pl_U = Delta_pl_N

        elif RHS_N*RHS_U<0:
            Delta_pl_L = Delta_pl_N
            Delta_pl_U = Delta_pl_U
                
                
    return Delta_pl*timeincrement, mat_params, q+1, sigma_star_N

In [12]:
timeincrement

1e-05

In [13]:
def implicit_integration_vp_dam(mat_params, i, N, timeincrement, number_of_iteration, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,stress_sph,l,m,n, N_dirr, sig_tr_i_1):
    

    Delta_pl_L  = 0
    Delta_pl_U1 = ((((stress_eqv_trial/(1)) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])**(N))*timeincrement
    #Delta_pl_U1 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial-mat_params['stress_yield'+str(i+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(m+1)+str(n+1)]))
    mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
    stress_dev_trial_n  =   mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-I*(1/3*mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].trace())
    J_stress_dev_n      =   np.sqrt(1.5*np.tensordot(stress_dev_trial_n, stress_dev_trial_n))
    stress_eqv_trial_n  = (np.sqrt(1.5*(np.tensordot(stress_dev_trial_n - stress_back_trial, stress_dev_trial-stress_back_trial))))/(1)
    stress_h            = 1/3*mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].trace()
    Delta_pl_U2 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))) + ((stress_eqv_trial_n/(1 - mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))-mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
    #Delta_pl_U2 = Delta_pl_U2/timeincrement
    strain_trial = tens_to_voigt(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
    strain_dev, strain_sph = split(strain_trial)
    sph_                   = ((K*strain_sph).trace()/3)
    
    #sph_        = (1/3)*(stress_sph).trace()
    #N_L         =  (1.5/(1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))*(stress_dev_trial - stress_back_trial)/stress_eqv_trial

    if Delta_pl_U1<Delta_pl_U2:

        Delta_pl_U = Delta_pl_U2
        #print('Delta_pl_U1 is chosen')

    else:
        Delta_pl_U = Delta_pl_U1
        #print('Delta_pl_U2 is chosen')

    print(Delta_pl_L, Delta_pl_U)
    
    D_L                    = (mat_params['D_cr'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]/(mat_params['e_r'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - mat_params['e_d'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))*((3*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*(stress_h/J_stress_dev_n)**(2))+((2/3)*(1+mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))*(Delta_pl_L)
    D_U                    = (mat_params['D_cr'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]/(mat_params['e_r'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - mat_params['e_d'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))*((3*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*(stress_h/J_stress_dev_n)**(2))+((2/3)*(1+mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))*(Delta_pl_U)
    
    #D_L = 0.0
    #D_U = 0.1
    
    #print(D_L, D_U)
    

    for q in range(1000):


        #sigma_star_L = stress_dev_trial - 3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_L*(N_dirr)*(timeincrement))
        
##################  Die Verarbeitung der Information fuer die erste Annahme ######################################
        
        
        
#         sigma_L      =  sig_tr_i_1 - (mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])).dot(N_L*Delta_pl_L*timeincrement)
#         sigma_star_L,sigma_sph_L =  split(sigma_L)
#         sigma_h_L    = (1/3)*(sigma_sph_L.trace())
#         X_star_L     = (stress_back_trial+ ((2500)*Delta_pl_L*timeincrement*(N_dirr)))/(1 + 20.30*Delta_pl_L*timeincrement) 
#         J2_L         = np.sqrt(1.5*np.tensordot(sigma_star_L - X_star_L, sigma_star_L-X_star_L))
#         R_L          = (mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_L*timeincrement)/(1 + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_L*timeincrement)
#         D_L          = (1) - (mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (1.35)*(Delta_pl_L*timeincrement)*((2/3)*(1+mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) + 
#                         3*((sph_/J2_L)**(2))*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))
#         RHS_L        = J2_L/(1-D_L) - (Delta_pl_L)**((1/N))*(mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-R_L
        
        h_L                      =         mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(Delta_pl_L/timeincrement)**(1/N) + (mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_L)/(1 + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_L)
#         D_L                      =         mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (1.35)*((2/3)*(1 + mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) 
#                                                                                                      +(3*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))*((sph_/h_L)**(2)))
        Z_L                      =         (((2*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(strain_dev)))*(1-D_L) - ((mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(1 + 20.30*Delta_pl_L))) 
        
        alpha_L                  =         1 + ((Delta_pl_L)/(h_L))*((3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(1)) + 2500/((1 + 20.30*(Delta_pl_L))*(1-D_L)))
        J_L                      =         np.sqrt(1.5*np.tensordot(Z_L, Z_L))
        RHS_L                    =         (J_L/alpha_L)*(1/(1-D_L)) - h_L
        
        stress_dev_L             =         (1-D_L)*2*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*strain_dev - 3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_L*(J_L/alpha_L)*Z_L
        J_stress_dev_L           =         np.sqrt(3/2*(np.tensordot(stress_dev_L, stress_dev_L)))                                                                                                                                                                        
        
        
        RHS_DL                   =         D_L - mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - (mat_params['D_cr'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]/(mat_params['e_r'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - mat_params['e_d'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))*(Delta_pl_L)*(((sph_*(1-D_L))**(2)/(J_stress_dev_L)**(2))*3*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) + (2/3)*(1 + mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
        
        
        
        
        h_U                      =         mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(Delta_pl_U/timeincrement)**(1/N) + (mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_U)/(1 + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_U)
        #D_U                      =         mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (1.35)*((2/3)*(1 + mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) 
        #                                                                                             +(3*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))*((sph_/h_U)**(2)))
        Z_U                      =         (2*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*((strain_dev)))*(1-D_U) - ((mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(1 + 20.30*Delta_pl_U)) 
        alpha_U                  =         1 + ((Delta_pl_U)/(h_U))*((3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(1)) + 2500/((1 + 20.30*(Delta_pl_U))*(1-D_U)))
        J_U                      =         np.sqrt(1.5*np.tensordot(Z_U, Z_U))
        RHS_U                    =         (J_U/alpha_U)*(1/(1-D_U)) - h_U
        
        stress_dev_U             =         (1-D_U)*2*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*strain_dev - 3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_U*(J_U/alpha_U)*Z_U
        J_stress_dev_U           =         np.sqrt(3/2*(np.tensordot(stress_dev_U, stress_dev_U)))
        RHS_DU                   =         D_U - mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - (mat_params['D_cr'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]/(mat_params['e_r'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - mat_params['e_d'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))*(Delta_pl_U)*(((sph_*(1-D_U))**2/(J_stress_dev_U)**2)*3*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) + (2/3)*(1 + mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
        
                                                         
                                                         
        Delta_pl_N               =         Delta_pl_U - (RHS_U*(Delta_pl_L-Delta_pl_U))/(RHS_L - RHS_U)
        D_N                      =         D_U        - (RHS_DU*(D_L - D_U)/(RHS_DL - RHS_DU))                                                 
        
        
        h_N                      =         mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(Delta_pl_N/timeincrement)**(1/N) + (mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_N)/(1 + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_N)
        #D_N                      =         mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (1.35)*((2/3)*(1 + mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) 
        #                                                                                             +(3*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))*((sph_/h_N)**(2)))
        Z_N                      =         (2*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(strain_dev))*(1-D_N) - ((mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(1 + 20.30*Delta_pl_N)) 
        alpha_N                  =         1 + ((Delta_pl_N)/(h_N))*((3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(1)) + 2500/((1 + 20.30*(Delta_pl_N))*(1-D_N)))
        J_N                      =         np.sqrt(1.5*np.tensordot(Z_N, Z_N))
        RHS_N                    =         (J_N/alpha_N)*(1/(1-D_N)) - h_N

        stress_dev_N             =         (1-D_N)*2*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*strain_dev - 3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_N*(J_N/alpha_N)*Z_N
        J_stress_dev_N           =         np.sqrt(3/2*(np.tensordot(stress_dev_N, stress_dev_N)))
        RHS_DN                   =         D_N - mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - (mat_params['D_cr'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]/(mat_params['e_r'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - mat_params['e_d'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))*(Delta_pl_N)*(((sph_*(1-D_N))**2/(J_stress_dev_N)**2)*3*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) + (2/3)*(1 + mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
        
        
        #print(RHS_DL, RHS_DU, RHS_DN, 'Schaedigung')
        #print(RHS_L, RHS_U, RHS_N, 'Plastiche Dehnung')
        
        
        
        #sigma_star_U = stress_dev_trial - 3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_U*(N_dirr)*(timeincrement))
        
#         sigma_U      = sig_tr_i_1 - (mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])).dot(N_L*Delta_pl_U*timeincrement)
#         sigma_star_U,sigma_sph_U =  split(sigma_U)
#         sigma_h_U    = (1/3)*(sigma_sph_U.trace())
#         X_star_U     = (stress_back_trial+ ((2500)*Delta_pl_U*timeincrement*(N_dirr)))/(1 + 20.30*Delta_pl_U*timeincrement) 
#         J2_U         = np.sqrt(1.5*np.tensordot(sigma_star_U - X_star_U, sigma_star_U-X_star_U))
#         R_U          = (mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_U*timeincrement)/(1 + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_U*timeincrement)
#         D_U          = (1) - (mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (1.35)*(Delta_pl_U*timeincrement)*((2/3)*(1+mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) + 
#                                      3*((sph_/J2_U)**(2))*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))
#         RHS_U        = J2_U/(1) - (Delta_pl_U)**((1/N))*(mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - R_U
#         Delta_pl_N   = Delta_pl_U - (RHS_U*(Delta_pl_L-Delta_pl_U))/(RHS_L - RHS_U)
#         sigma_star_N = stress_dev_trial - 3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_N*(N_dirr)*(timeincrement))
#         X_star_N     = (stress_back_trial+ ((2500)*Delta_pl_N*timeincrement*(N_dirr)))/(1 + 20.30*Delta_pl_N*timeincrement) 
#         J2_N         = np.sqrt(1.5*np.tensordot(sigma_star_N - X_star_N, sigma_star_N-X_star_N))
#         R_N          = mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*Delta_pl_N*timeincrement
#         #D_N          = (1) - (mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (1.35)*(Delta_pl_N*timeincrement)*((2/3)*(1+mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) + 
#         #                             3*((sph_/J2_N)**(2))*(1-2*mat_params['pois'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))
#         RHS_N        = J2_N/(1) - (Delta_pl_N)**((1/N))*(mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - R_N
        #print(RHS_DL, RHS_DU, RHS_DN, D_N, 'Schaedigung')
        #print(RHS_L, RHS_U, RHS_N, 'Plastische Dehnung')
    
        if abs(RHS_N)<1e-4 and abs(RHS_DN)<1e-4 :

            Delta_pl = Delta_pl_N
            mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]        = D_N
            mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]        = (mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_N)/(1 + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_N)
            #if sequence_tracker==0:
            #    deqpl1.append(Delta_pl)
            #elif sequence_tracker==1:
            #    deqpl2.append(Delta_pl)
            #print('In the converged zone')
            print('Your increment number '+str(t+1)+' has converged with '+str(q+1)+' iterations')
            
            
            break

#         if RHS_N*RHS_L<0:

#             Delta_pl_L = Delta_pl_L
#             Delta_pl_U = Delta_pl_N
        
        
        if RHS_DN*RHS_DL<0:
            
            D_L        = D_L
            D_U        = D_N
            #print('Eingetreten I', D_L, D_U)

        if RHS_DN*RHS_DU<0:
            
            D_L        = D_N
            D_U        = D_U
            #print('Eingetreten II', D_L, D_U)
            

#         if RHS_N*RHS_U<0:
#             Delta_pl_L = Delta_pl_N
#             Delta_pl_U = Delta_pl_U
                
                
    return Delta_pl, mat_params, q+1, (((Z_N/alpha_N)/J_N)*(1/(1-D_N))), D_N

In [14]:
# fe_sol_strain = []
# fe_sol_stress = []
# fe_sol_strain.append(np.genfromtxt(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\Plate paper with damage implicit viscoplasticity\BVP2\Stress-Strain-Data\without_hole_strain'+str(26)+str(1)+str(1)+str(1)+'.csv'))
# fe_sol_stress.append(np.genfromtxt(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\Plate paper with damage implicit viscoplasticity\BVP2\Stress-Strain-Data\without_hole_stress'+str(26)+str(1)+str(1)+str(1)+'.csv'))

In [15]:
# delta_st = np.array([-1.85126535e-04,  6.01295524e-06,  8.42887387e-05,  2.00457594e-05,
#        -3.08982564e-05,  1.11731923e-05])*timeincrement

In [16]:
delta_st = delta_st = np.array([-5.28392860e-07,  1.77573594e-09,  2.47819813e-07, -2.54091805e-07,
        1.45216080e-07, -2.62235239e-08])*1e-1

In [17]:
l = 0
m = 0
n = 0
random   = 0
Delta_pl = 0
wahl    = 0
deqpl   = []
deqpl1           = []
deqpl2           = []
eps33_ana        = []
eps33_ana_pl     = []
count_tr         = 0
sequence_tracker = 0
trial_stress = []
back_stress  = []
number_of_iteration = 0
start_time = datetime.now()

sign   =  [-1]

for ll in range(len(sign)):
    
    print('......................................................................................................')
    print('Load Sequence '+str(l+1)+' The sign is '+str(sign[l]))
    print('......................................................................................................')
    
    for t in range(50000):


        mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]                    = delta_st*1*sign[ll]
        
        #print(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])

        for l in range((1)):

            for m in range((1)):

                for n in range((1)):

                    mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]      = (mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy())*(1)
                    #sig_tr_i_1                                                        = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    + ((mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*((1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))).dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
                    sig_tr_i_1                                                        = mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(tens_to_voigt(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
                    mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = (mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy())*(1)
                    #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    + mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps_cor'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) 
                    #mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])  
                    stress_back_trial                                                 = mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy()
                    mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                    #mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                    eps_trial_                                                        = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy()
                    stress_dev_trial,stress_sph                                       = split(sig_tr_i_1)
                    #if i==0 and m==0 and n==0:
                    #    print(mat_params['stress_n111'])
                    zeta                                                              = stress_dev_trial-stress_back_trial
                    stress_eqv_trial                                                  = np.sqrt(1.5*(np.tensordot(stress_dev_trial-stress_back_trial, stress_dev_trial-stress_back_trial)))    
                    N_dirr                                                            = ((stress_dev_trial - stress_back_trial)/stress_eqv_trial)


                    # Write a function to give the current yield stress
                    #mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['H'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = Uhard(mat_params['EQPLAS'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                    #print(mat_params)

                    if (stress_eqv_trial/((1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))) > abs(mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]+mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]):

                        #print('Hello')
                        if count_tr==0 and mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][0][0]==0:
                            sequence_tracker=0
                            count_tr=1

                        elif count_tr==0 and mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][0][0]!=0:
                            sequence_tracker=1
                            count_tr=1

                        for p in range(5):

                            if not(p==0):

                                #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    + mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps_cor'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                                #mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])  
                                #mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                                #stress_dev_trial                                              = mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-I*(1/3*mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].trace())
                                #if i==0 and m==0 and n==0:

                                #print('hello')
                                #    print(mat_params['stress_n111'])
                                sig_tr_i_1                                                    = mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    + (mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(1 - mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])).dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                                stress_dev_trial,stress_sph                                   = split(sig_tr_i_1)
                                stress_eqv_trial                                              = np.sqrt(1.5*(np.tensordot(stress_dev_trial-stress_back_trial, stress_dev_trial-stress_back_trial)))    
                                N_dirr                                                        = (stress_dev_trial-stress_back_trial)/stress_eqv_trial

                                sigma_33_trial                                                = sig_tr_i_1[2]
                                #Delta_pl, mat_params, q, sigma_star_N                         = implicit_integration_vp_1(mat_params, i, N, timeincrement, 1, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n,N_dirr)
                                #Delta_pl, mat_params, q, sigma_star_N, J2_N                   = implicit_integration_vp_3_dam(mat_params, i, N, timeincrement, number_of_iteration, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,stress_sph,l,m,n, N_dirr, sigma_)
                                Delta_pl, mat_params, q, N_plas,D_N   = implicit_integration_vp_dam(mat_params, i, N, timeincrement, number_of_iteration, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,stress_sph,l,m,n, N_dirr, sig_tr_i_1)
                                sigma_i_1                                                     = sigma_33_trial - ((3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl*((stress_dev_trial[2][2]-stress_back_trial[2][2]))/stress_eqv_trial))
                                delta_eps33_i_1                                               = delta_eps33_i_minus_1 -(((sigma_i)*(delta_eps33_i - delta_eps33_i_minus_1))/(1e-9 + sigma_i_1 - sigma_i))


    #                             a1r, b1r, c1r, d1r, e1r, f1r     =   ML.tens_to_voigt(stress_dev_trial)
    #                             a2r, b2r, c2r, d2r, e2r, f2r     =   ML.tens_to_voigt(stress_back_trial)
    #                             trial_stress.append([a1r, b1r, c1r, d1r, e1r, f1r])
    #                             back_stress.append([a2r, b2r, c2r, d2r, e2r, f2r])
    #                             if sequence_tracker==0:
    #                                 deqpl1.append(Delta_pl)

    #                             elif sequence_tracker==1:

    #                                 deqpl2.append(Delta_pl)

                                if abs(delta_eps33_i - delta_eps33_i_minus_1)/abs(delta_eps33_i_1) < 1e-8:

                                    print('Wir befinden uns weiter im Nichtlinearen Bereich und Konvergieren in '+str(p+1)+' Iterations')   
                                    deqpl1.append(Delta_pl)
                                    a1r, b1r, c1r, d1r, e1r, f1r     =   ML.tens_to_voigt(stress_dev_trial)
                                    a2r, b2r, c2r, d2r, e2r, f2r     =   ML.tens_to_voigt(stress_back_trial)
                                    trial_stress.append([a1r, b1r, c1r, d1r, e1r, f1r])
                                    back_stress.append([a2r, b2r, c2r, d2r, e2r, f2r])
                                    mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2] = delta_eps33_i_1
                                    #eps33_ana.append(delta_eps33_i_1)
                                    break

                                #print('Trial_Stress '+str(sigma_33_trial),'Sigma_i_1 ' +str(sigma_i_1), 'Equivalent Plastic Strain: '+str(Delta_pl))
                                mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2] = delta_eps33_i_1
                                delta_eps33_i_minus_1 =  delta_eps33_i
                                delta_eps33_i         =  delta_eps33_i_1
                                sigma_i               =  sigma_i_1



                            elif p==0:

    #                             a1r, b1r, c1r, d1r, e1r, f1r     =   ML.tens_to_voigt(stress_dev_trial)
    #                             a2r, b2r, c2r, d2r, e2r, f2r     =   ML.tens_to_voigt(stress_back_trial)
    #                             trial_stress.append([a1r, b1r, c1r, d1r, e1r, f1r])
    #                             back_stress.append([a2r, b2r, c2r, d2r, e2r, f2r])





                                sigma_33_trial                         = sig_tr_i_1[2]
                                #mat_params, Delta_pl  = implicit_integration(mat_params, stress_eqv_trial,i,l,m,n)
                                #Delta_pl, mat_params, q, sigma_star_N  = implicit_integration_vp_1(mat_params, i, N, timeincrement, 1, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n,N_dirr)
                                Delta_pl, mat_params, q, N_plas,D_N   = implicit_integration_vp_dam(mat_params, i, N, timeincrement, number_of_iteration, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,stress_sph,l,m,n, N_dirr, sig_tr_i_1)
                                sigma_i_1                              = sigma_33_trial - ((3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*((1))*Delta_pl*(stress_dev_trial[2][2]-stress_back_trial[2][2]))/stress_eqv_trial)
                                sigma_i                                = sigma_i_1
                                delta_eps33_i_minus_1                  = mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2] 
                                mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2] = -(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][0] + mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][1])
                                delta_eps33_i                          = mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2]
    #                             if sequence_tracker==0:
    #                                 deqpl1.append(Delta_pl)

    #                             elif sequence_tracker==1:
    #                                 deqpl2.append(Delta_pl)
                                #print(delta_eps33_i, delta_eps33_i_minus_1, Delta_pl)
                        #Delta_pl    = 0
                        #mat_params['flag'+str(i+1)]+=1
                        #print(mat_params['flag'+str(i+1)])
                        #delta_ep    = 0

        #                     for j in range(1000):


        #                         RHS                              = stress_eqv_trial  - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] -  3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(Delta_pl))
        #                         #a                                = RHS/(mat_params['H'+str(i+1)] + (mat_params['E'+str(i+1)]))
        #                         #print(mat_params['H'+str(i+1)])
        #                         Delta_pl                         = Delta_pl +  (RHS/(mat_params['H'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + 3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
        #                         #delta_ep                         = delta_ep + (Delta_pl)
        #                         #print(Delta_pl)
        #                         #mat_params['EQPLAS'+str(i+1)]    = mat_params['EQPLAS'+str(i+1)] + Delta_pl

        #                         mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['H'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = Uhard(mat_params['EQPLAS'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]+Delta_pl)
        #                         #print(mat_params['H'+str(i+1)])
        #                         #print(Delta_pl, mat_params['EQPLAS'+str(i+1)], RHS)
        #                         #mat_params['C'+str(i+1)][0][0]                                = (mat_params['E'+str(i+1)]*mat_params['H'+str(i+1)])/(mat_params['E'+str(i+1)] + mat_params['H'+str(i+1)])
        #                         #print(RHS, mat_params['stress_yield'+str(i+1)])
        #                         if abs(stress_eqv_trial  - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] -  3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(Delta_pl)))<1e-5:
        #                             #print(j+1, stress_eqv_trial  - mat_params['stress_yield'+str(i+1)] -  (3*mat_params['mu'+str(i+1)]*(Delta_pl)))
        #                             #print('....Converged...... in '+str(j+1)+' iterations')
        #                             #print(Delta_pl)
        #                             break

                        # Now we move ahead with updating the stress, elastic and plastic strains
                        #print('...............out from the loop.......')

                        #if mat_params['flag'+str(i+1)]==0:
                        #    mat_params['stress_tens'+str(i+1)]        = ((stress_dev_trial/(stress_eqv_trial))*mat_params['stress_yield'+str(i+1)]) + I*(ss) 
                        #    print(mat_params['stress_tens'+str(i+1)])
                        #    mat_params['flag'+str(i+1)]               +=1
                        #    mat_params['stress_tens'+str(i+1)]        = ((stress_dev_trial/(stress_eqv_trial))*mat_params['stress_yield'+str(i+1)]) + I*(ss)  + np.array([[4.2,0,0],[0,0,0],[0,0,0]])
                        #    print(mat_params['stress_tens'+str(i+1)])

                        #else:
                        #if i==0 and m==0 and n==0:

                        #    print((mat_params['stress_n111'] - ML.tens_to_voigt(2*mat_params['mu111']*(1.5)*Delta_pl*(stress_dev_trial/(stress_eqv_trial))))[0])
                        #mat_params['stress_yield'+str(i+1)],_ = Uhard(mat_params['EQPLAS'+str(i+1)]+Delta_pl)
                        #dummy_sph                             = (stress_dev_trial/(stress_eqv_trial))
                        #mat_params['stress_yield'+str(i+1)],_ = Uhard(mat_params['EQPLAS'+str(i+1)]+Delta_pl)
                        #mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    = ((stress_dev_trial/(stress_eqv_trial))*(mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])) + I*(mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*dummy_sph[0][0]/(3*abs(dummy_sph[0][0])))
                        #mat_params['stress_tens'+str(i+1)]    = ((stress_dev_trial/(stress_eqv_trial))*(mat_params['stress_yield'+str(i+1)])) + I*(mat_params['stress_yield'+str(i+1)]/3)
                        #print(((stress_dev_trial/(stress_eqv_trial))*mat_params['stress_yield'+str(i+1)]))

                        #print('.............................')
                        #print()
                        #print(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2], delta_st[0][2])
                        #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]  = ML.tens_to_voigt(mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        #sigma_h                                                       = stress_sph.trace()/3
                        #mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        a1                                                            = ML.tens_to_voigt(mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        b1                                                            = ML.tens_to_voigt(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        c1                                                            = ML.tens_to_voigt(N_plas)

                        for j in range(3):

                            a1[j]                                 = a1[j]  +  (1.5*(Delta_pl)*(c1[j]/1))
                            b1[j]                                 = b1[j]  -  (1.5*(Delta_pl)*(c1[j]/1))

                        for j in range(3,6):

                            a1[j]                                 = a1[j]  +  (3*(Delta_pl)*(c1[j]/1))
                            b1[j]                                 = b1[j]  -  (3*(Delta_pl)*(c1[j]/1))


                        #mat_params['stress_n'+str(i+1)]           = mat_params['stress_n'+str(i+1)] + mat_params['C_ep'+str(i+1)].dot(mat_params['delta_eps'+str(i+1)]-1.5*(Delta_pl)*(c1/stress_eqv_trial))
                        #Delta_X                                                              =  (2/3)*(3100)*(Delta_pl*3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*((zeta)/stress_eqv_trial)) - 110*(stress_back_trial)*Delta_pl
                        #X_n                                                                  = (mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy())*(1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        #eps_trial_                                                           = tens_to_voigt(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]        = ML.voigt_to_tens(a1)
                        mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]        = ML.voigt_to_tens(b1)
                        mat_params['EQPLAS'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]             = mat_params['EQPLAS'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + Delta_pl
                        #mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]                  = mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]      + mat_params['b1'+str(i+1)+str(l+1)+s   
                        mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]                  = (mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl)/(1     +      Delta_pl*mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        
                        #mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]                  = D_N                    
                        
                        if mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]>0.45:
                            print('SOS: Das Material wurde geschadet!!!!!!!!!!!!!')
                            break
                        
                        mat_params['flow'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]               = ML.tens_to_voigt(N_plas)
                        #int_1                                                                = stress_dev_trial - (Delta_pl*ML.voigt_to_tens(mat_params['flow'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))*3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]
                        #int_2                                                                = K*I*(ML.voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])).trace()*(1/3)
                        #N_dirr                                                               = stress_dev_trial-stress_back_trial/stress_eqv_trial
                        #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]           = mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - (Delta_pl*mat_params['flow'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]                                          
                        #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]           = tens_to_voigt((1/3)*(ML.voigt_to_tens(mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])).trace()*I + int_1 + int_2)
                        #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]            = mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]/(1) + mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-(1.5/(1-D_N))*Delta_pl*mat_params['flow'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) 
                        mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]            = (1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-(1.5*Delta_pl*c1)) + mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]                                                                     
                        #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]            = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]/(1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        #mat_params['flow'+str(i+1)]               = mat_params['flow'+str(i+1)].reshape(6,1)
                        #mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]        = ((mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (2500*Delta_pl*(1.5/(1-D_N))*N_plas))/(1 + 20.30*Delta_pl))/(1) 
                        mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]         = (mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    +  2500*(Delta_pl)*N_plas)/(1   +   Delta_pl*20.30)
                        #mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2][2]  =  0
                        #mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]       = (mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + Delta_X)
                        #print(mat_params['EQPLAS'+str(i+1)], Delta_pl)
                        # Updating the material jacobian with a function
                        mat_params['stress_eqv_trial'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]   = stress_eqv_trial
                        #mat_params['C_ep'+str(i+1)+str(l+1)+str(m+1)+str(n+1)],mat_params    = material_jacobian_evp(mat_params, i, N, N_dirr, I, l, m, n, Delta_pl, alpha_N*h_N, index, stress_dev_trial, timeincrement)
                        #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2]        = 0 
                        mat_params['abs_strain'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(ML.tens_to_voigt(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
                        mat_params['abs_stress'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append((mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
                        #print(mat_params['EQPLAS'+str(i+1)+str(m+1)+str(n+1)])
                        mat_params['abs_eqpl'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(Delta_pl)
                        #print('Stress_yield through material law:',mat_params['stress_yield'+str(i+1)+str(m+1)+str(n+1)])
                        eps33_ana_pl.append(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2][2] + mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2][2])
                        #print('....................................')
                        #print(mat_params['EQPLAS1'])

                    else:
                        
                        #mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]        = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]           = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]  + (mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(1)).dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        #print(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]           = (1-mat_params['D'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(tens_to_voigt(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
                        mat_params['stress_eqv_trial'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]   = stress_eqv_trial
                        mat_params['C_ep'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]               = mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy()
                        mat_params['abs_strain'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(ML.tens_to_voigt(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
                        mat_params['abs_stress'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        eps33_ana.append(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2][2] + mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2][2])
                        #print(mat_params['stress_n1111'])
                        print('Wir befinden uns im Linearen bereich des Zeitinkrements: '+str(t+1))
                        #count_tr = 0
                    
print('Duration: {}'.format(datetime.now() - start_time))

......................................................................................................
Load Sequence 1 The sign is -1
......................................................................................................
Wir befinden uns im Linearen bereich des Zeitinkrements: 1
Wir befinden uns im Linearen bereich des Zeitinkrements: 2
Wir befinden uns im Linearen bereich des Zeitinkrements: 3
Wir befinden uns im Linearen bereich des Zeitinkrements: 4
Wir befinden uns im Linearen bereich des Zeitinkrements: 5
Wir befinden uns im Linearen bereich des Zeitinkrements: 6
Wir befinden uns im Linearen bereich des Zeitinkrements: 7
Wir befinden uns im Linearen bereich des Zeitinkrements: 8
Wir befinden uns im Linearen bereich des Zeitinkrements: 9
Wir befinden uns im Linearen bereich des Zeitinkrements: 10
Wir befinden uns im Linearen bereich des Zeitinkrements: 11
Wir befinden uns im Linearen bereich des Zeitinkrements: 12
Wir befinden uns im Linearen bereich des Zeitinkreme

C:\Users\Tandale\AppData\Local\Temp/ipykernel_21328/1553325046.py:55: RuntimeWarning: invalid value encountered in true_divide
  N_dirr                                                            = ((stress_dev_trial - stress_back_trial)/stress_eqv_trial)



Wir befinden uns im Linearen bereich des Zeitinkrements: 1533
Wir befinden uns im Linearen bereich des Zeitinkrements: 1534
Wir befinden uns im Linearen bereich des Zeitinkrements: 1535
Wir befinden uns im Linearen bereich des Zeitinkrements: 1536
Wir befinden uns im Linearen bereich des Zeitinkrements: 1537
Wir befinden uns im Linearen bereich des Zeitinkrements: 1538
Wir befinden uns im Linearen bereich des Zeitinkrements: 1539
Wir befinden uns im Linearen bereich des Zeitinkrements: 1540
Wir befinden uns im Linearen bereich des Zeitinkrements: 1541
Wir befinden uns im Linearen bereich des Zeitinkrements: 1542
Wir befinden uns im Linearen bereich des Zeitinkrements: 1543
Wir befinden uns im Linearen bereich des Zeitinkrements: 1544
Wir befinden uns im Linearen bereich des Zeitinkrements: 1545
Wir befinden uns im Linearen bereich des Zeitinkrements: 1546
Wir befinden uns im Linearen bereich des Zeitinkrements: 1547
Wir befinden uns im Linearen bereich des Zeitinkrements: 1548
Wir bef

Wir befinden uns im Linearen bereich des Zeitinkrements: 2968
Wir befinden uns im Linearen bereich des Zeitinkrements: 2969
Wir befinden uns im Linearen bereich des Zeitinkrements: 2970
Wir befinden uns im Linearen bereich des Zeitinkrements: 2971
Wir befinden uns im Linearen bereich des Zeitinkrements: 2972
Wir befinden uns im Linearen bereich des Zeitinkrements: 2973
Wir befinden uns im Linearen bereich des Zeitinkrements: 2974
Wir befinden uns im Linearen bereich des Zeitinkrements: 2975
Wir befinden uns im Linearen bereich des Zeitinkrements: 2976
Wir befinden uns im Linearen bereich des Zeitinkrements: 2977
Wir befinden uns im Linearen bereich des Zeitinkrements: 2978
Wir befinden uns im Linearen bereich des Zeitinkrements: 2979
Wir befinden uns im Linearen bereich des Zeitinkrements: 2980
Wir befinden uns im Linearen bereich des Zeitinkrements: 2981
Wir befinden uns im Linearen bereich des Zeitinkrements: 2982
Wir befinden uns im Linearen bereich des Zeitinkrements: 2983
Wir befi

Wir befinden uns im Linearen bereich des Zeitinkrements: 4683
Wir befinden uns im Linearen bereich des Zeitinkrements: 4684
Wir befinden uns im Linearen bereich des Zeitinkrements: 4685
Wir befinden uns im Linearen bereich des Zeitinkrements: 4686
Wir befinden uns im Linearen bereich des Zeitinkrements: 4687
Wir befinden uns im Linearen bereich des Zeitinkrements: 4688
Wir befinden uns im Linearen bereich des Zeitinkrements: 4689
Wir befinden uns im Linearen bereich des Zeitinkrements: 4690
Wir befinden uns im Linearen bereich des Zeitinkrements: 4691
Wir befinden uns im Linearen bereich des Zeitinkrements: 4692
Wir befinden uns im Linearen bereich des Zeitinkrements: 4693
Wir befinden uns im Linearen bereich des Zeitinkrements: 4694
Wir befinden uns im Linearen bereich des Zeitinkrements: 4695
Wir befinden uns im Linearen bereich des Zeitinkrements: 4696
Wir befinden uns im Linearen bereich des Zeitinkrements: 4697
Wir befinden uns im Linearen bereich des Zeitinkrements: 4698
Wir befi

Wir befinden uns im Linearen bereich des Zeitinkrements: 6245
Wir befinden uns im Linearen bereich des Zeitinkrements: 6246
Wir befinden uns im Linearen bereich des Zeitinkrements: 6247
Wir befinden uns im Linearen bereich des Zeitinkrements: 6248
Wir befinden uns im Linearen bereich des Zeitinkrements: 6249
Wir befinden uns im Linearen bereich des Zeitinkrements: 6250
Wir befinden uns im Linearen bereich des Zeitinkrements: 6251
Wir befinden uns im Linearen bereich des Zeitinkrements: 6252
Wir befinden uns im Linearen bereich des Zeitinkrements: 6253
Wir befinden uns im Linearen bereich des Zeitinkrements: 6254
Wir befinden uns im Linearen bereich des Zeitinkrements: 6255
Wir befinden uns im Linearen bereich des Zeitinkrements: 6256
Wir befinden uns im Linearen bereich des Zeitinkrements: 6257
Wir befinden uns im Linearen bereich des Zeitinkrements: 6258
Wir befinden uns im Linearen bereich des Zeitinkrements: 6259
Wir befinden uns im Linearen bereich des Zeitinkrements: 6260
Wir befi

... (2 outputs left out)

Wir befinden uns im Linearen bereich des Zeitinkrements: 10847
Wir befinden uns im Linearen bereich des Zeitinkrements: 10848
Wir befinden uns im Linearen bereich des Zeitinkrements: 10849
Wir befinden uns im Linearen bereich des Zeitinkrements: 10850
Wir befinden uns im Linearen bereich des Zeitinkrements: 10851
Wir befinden uns im Linearen bereich des Zeitinkrements: 10852
Wir befinden uns im Linearen bereich des Zeitinkrements: 10853
Wir befinden uns im Linearen bereich des Zeitinkrements: 10854
Wir befinden uns im Linearen bereich des Zeitinkrements: 10855
Wir befinden uns im Linearen bereich des Zeitinkrements: 10856
Wir befinden uns im Linearen bereich des Zeitinkrements: 10857
Wir befinden uns im Linearen bereich des Zeitinkrements: 10858
Wir befinden uns im Linearen bereich des Zeitinkrements: 10859
Wir befinden uns im Linearen bereich des Zeitinkrements: 10860
Wir befinden uns im Linearen bereich des Zeitinkrements: 10861
Wir befinden uns im Linearen bereich des Zeitinkrements

Wir befinden uns im Linearen bereich des Zeitinkrements: 12448
Wir befinden uns im Linearen bereich des Zeitinkrements: 12449
Wir befinden uns im Linearen bereich des Zeitinkrements: 12450
Wir befinden uns im Linearen bereich des Zeitinkrements: 12451
Wir befinden uns im Linearen bereich des Zeitinkrements: 12452
Wir befinden uns im Linearen bereich des Zeitinkrements: 12453
Wir befinden uns im Linearen bereich des Zeitinkrements: 12454
Wir befinden uns im Linearen bereich des Zeitinkrements: 12455
Wir befinden uns im Linearen bereich des Zeitinkrements: 12456
Wir befinden uns im Linearen bereich des Zeitinkrements: 12457
Wir befinden uns im Linearen bereich des Zeitinkrements: 12458
Wir befinden uns im Linearen bereich des Zeitinkrements: 12459
Wir befinden uns im Linearen bereich des Zeitinkrements: 12460
Wir befinden uns im Linearen bereich des Zeitinkrements: 12461
Wir befinden uns im Linearen bereich des Zeitinkrements: 12462
Wir befinden uns im Linearen bereich des Zeitinkrements

Wir befinden uns im Linearen bereich des Zeitinkrements: 13960
Wir befinden uns im Linearen bereich des Zeitinkrements: 13961
Wir befinden uns im Linearen bereich des Zeitinkrements: 13962
Wir befinden uns im Linearen bereich des Zeitinkrements: 13963
Wir befinden uns im Linearen bereich des Zeitinkrements: 13964
Wir befinden uns im Linearen bereich des Zeitinkrements: 13965
Wir befinden uns im Linearen bereich des Zeitinkrements: 13966
Wir befinden uns im Linearen bereich des Zeitinkrements: 13967
Wir befinden uns im Linearen bereich des Zeitinkrements: 13968
Wir befinden uns im Linearen bereich des Zeitinkrements: 13969
Wir befinden uns im Linearen bereich des Zeitinkrements: 13970
Wir befinden uns im Linearen bereich des Zeitinkrements: 13971
Wir befinden uns im Linearen bereich des Zeitinkrements: 13972
Wir befinden uns im Linearen bereich des Zeitinkrements: 13973
Wir befinden uns im Linearen bereich des Zeitinkrements: 13974
Wir befinden uns im Linearen bereich des Zeitinkrements

UnboundLocalError: local variable 'Delta_pl' referenced before assignment

In [ ]:
mat_params['stress_n-11111'] + mat_params['C1111'].dot(mat_params['delta_eps1111']) + tens_to_voigt(-3*mat_params['mu1111']*Delta_pl*(N_plas))

In [ ]:
tens_to_voigt(-3*mat_params['mu1111']*Delta_pl*(N_plas))

In [ ]:
stress_eqv_trial - mat_params['stress_K1111']*(Delta_pl/timeincrement)**(1/N)

In [ ]:
mat_params['R1111'] + mat_params['stress_yield1111']

In [ ]:
for i in range(5):
    
    mat_params['strain'+str(i+1)] = np.genfromtxt(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\SLMU_paper\Isotropic_Damage_data_sets_trial\mit_schaedigung_strain_'+str(i+1)+'.csv', delimiter=',')
    mat_params['stress'+str(i+1)] = np.genfromtxt(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\SLMU_paper\Isotropic_Damage_data_sets_trial\mit_schaedigung_stress_'+str(i+1)+'.csv', delimiter=',')

In [ ]:
# for i in range(5):
    
#     np.savetxt(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\SLMU_paper\Isotropic_Damage_data_sets_trial\mit_schaedigung_strain_'+str(i+1)+'.csv', np.array(mat_params['abs_strain1111'])[:,i:i+1], delimiter=',')
#     np.savetxt(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\SLMU_paper\Isotropic_Damage_data_sets_trial\mit_schaedigung_stress_'+str(i+1)+'.csv', np.array(mat_params['abs_stress1111'])[:,i:i+1], delimiter=',')

In [ ]:
fig = plt.figure(figsize=(12,9))
ax = fig.add_subplot(111)
ax.plot(np.array(mat_params['abs_strain1111'])[:,0:1], np.array(mat_params['abs_stress1111'])[:,0:1], label='$\sigma_{11}$ mit Schaedigung')
#ax.plot(np.array(mat_params['abs_strain1111'])[:,1:2], np.array(mat_params['abs_stress1111'])[:,1:2], label='Stress2')
# ax.plot(np.array(mat_params['abs_strain1111'])[:,2:3], np.array(mat_params['abs_stress1111'])[:,2:3], label='Stress3')
# ax.plot(np.array(mat_params['abs_strain1111'])[:,3:4], np.array(mat_params['abs_stress1111'])[:,3:4], label='Stress4')
# ax.plot(np.array(mat_params['abs_strain1111'])[:,4:5], np.array(mat_params['abs_stress1111'])[:,4:5], label='Stress5')
# ax.plot(np.array(mat_params['abs_strain1111'])[:,5:6], np.array(mat_params['abs_stress1111'])[:,5:6], label='Stress6')
#ax.plot(np.array(mat_params['strain1']), np.array(mat_params['stress1']), label='Stress_ohne_schaedigung')
# ax.plot(np.array(mat_params['strain2']), np.array(mat_params['stress2']), label='Stress_ohne_schaedigung')
# ax.plot(np.array(mat_params['strain3']), np.array(mat_params['stress3']), label='Stress_ohne_schaedigung')
# ax.plot(np.array(mat_params['strain4']), np.array(mat_params['stress4']), label='Stress_ohne_schaedigung')
# ax.plot(np.array(mat_params['strain5']), np.array(mat_params['stress5']), label='Stress_ohne_schaedigung')
#ax.scatter(np.linspace(1, len(deqpl1), len(deqpl1)), deqpl1)
#ax.scatter(np.linspace(1, len(deqpl2), len(deqpl2)), deqpl2)
# ax.scatter(np.linspace(1, len(eps33_ana), len(eps33_ana)), eps33_ana, label='Elastic')
# ax.scatter(np.linspace(len(eps33_ana), len(eps33_ana_pl)+len(eps33_ana), len(eps33_ana_pl)), eps33_ana_pl, label='Plastic')
# ax.scatter(np.linspace(1, el_.shape[0], el_.shape[0]), el_, label='Elastic_5e-4')
# ax.scatter(np.linspace(el_.shape[0], pl_.shape[0]+el_.shape[0], pl_.shape[0]), pl_, label='Elastic_5e-4')

plt.xlabel('TimeSteps')
plt.ylabel('Equivalent Plastic Strain')
plt.legend()
plt.show()


In [ ]:
# np.savetxt('Timeinc_5e-4_el.csv', np.array(eps33_ana))
# np.savetxt('Timeinc_5e-4_pl.csv', np.array(eps33_ana_pl))

# el_ = np.genfromtxt('Timeinc_5e-4_el.csv')
# pl_ = np.genfromtxt('Timeinc_5e-4_pl.csv')

In [ ]:
mat_params['D1111']

In [ ]:
mat_params['abs_strain1111'][0]*16000

In [ ]:
#np.savetxt('Einfach Stress', np.array(mat_params['abs_stress1111'])[1429:6000,:], delimiter=',')

In [ ]:
mat_params['stress_n1111']

In [ ]:
# dummy1 = np.concatenate((np.array(deqpl1)[0:len(deqpl2)], np.array(deqpl2)[0:len(deqpl2)]), axis=0)
    
# dummy2 = np.concatenate((np.array(trial_stress)[0:len(deqpl2), :], np.array(trial_stress)[len(deqpl1):len(trial_stress), :]), axis=0)
# dummy3 = np.concatenate((np.array(back_stress)[0:len(deqpl2), :], np.array(back_stress)[len(deqpl1):len(trial_stress), :]), axis=0)    

    
    

In [ ]:
# np.savetxt('Plate_Element_Trial_Stress', dummy2, delimiter=',')

# np.savetxt('Plate_Element_back_Stress', dummy3, delimiter=',')

# np.savetxt('Plate_Element_EQPL', dummy1, delimiter=',')


In [ ]:
mat_params['D1111']

In [ ]:
np.concatenate((mat_params['C1111'], mat_params['C1111']), axis=0).reshape(2,6,6)

In [ ]:
np.einsum('BNi,Bi ->BN', np.concatenate((mat_params['C1111'], mat_params['C1111']), axis=0).reshape(2,6,6), np.concatenate((delta_st[0], delta_st[0]), axis=0).reshape(2,6))

In [ ]:
mat_params['C1111'].dot(delta_st[0])

In [ ]:
np.concatenate((delta_st[0], delta_st[0]), axis=0).shape

In [ ]:
sig_tr_i_1

In [ ]:
mat_params['stress_n1111']

In [ ]:
mat_params['stress_n1111']

In [ ]:
mat_params['delta_eps1111']

In [ ]:
mat_params['C1111']

In [ ]:
mat_params['C1111'].dot(mat_params['delta_eps1111'])

In [ ]:
example = np.zeros(shape=(3,3))



In [ ]:
example[np.triu_indices(3, k = 0)] = example[np.triu_indices(3, k = 0)] + np.ones(shape=(6,))
#example[np.triu_indices(3, k = 0)] = example[np.triu_indices(3, k = 0)]  + np.ones(shape=(3,))

In [ ]:
example[np.triu_indices(3, k = 0)]

In [ ]:
example

In [ ]:
np.triu_indices(3, k = 0)

In [ ]:
mat_params['eps_el_tens1111'] + mat_params['eps_pl_tens1111']

In [ ]:
mat_params['abs_strain1111']

In [ ]:
N_dirr

In [ ]:
N_dirr*Delta_pl*1.5

In [ ]:
11.5*3

In [ ]:
mat_params['eps_']

In [ ]:
eps33_ana

In [ ]:
mat_params['delta_eps1111']*100*sign[1]

In [ ]:
mat_params['abs_strain1111']

In [ ]:
stress_eqv_trial

In [ ]:
stress_dev_trial-stress_back_trial

In [ ]:
sig_tr_i_1

In [ ]:
voigt_to_tens(sig_tr_i_1).trace()*I/3

In [ ]:
# np.savetxt('Strain Example', np.array(mat_params['abs_strain1111']), delimiter=',')
# np.savetxt('Stress Example', np.array(mat_params['abs_stress1111']), delimiter=',')

In [ ]:
#np.savetxt(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\Plate paper with damage implicit viscoplasticity\Single_Element_Case_Study\Time_increment_0.1\inp.csv', inpp,delimiter=',')

In [ ]:
#np.savetxt(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\Plate paper with damage implicit viscoplasticity\Single_Element_Case_Study\Time_increment_0.1\out.csv', np.array(deqpl1),delimiter=',')

In [ ]:
stress_eqv_trial

In [ ]:
stress_dev_trial

In [ ]:
stress_eqv_trial

In [ ]:
mat_params['abs_strain1111']

In [ ]:
ma

In [ ]:
plt.imshow()